In [1]:
from pathlib import Path

from imagematerials.__main__ import export_summary_netcdf, simulate_stocks
from imagematerials.vehicles.preprocessing import (
    preprocessing,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.factory import ModelFactory
import prism


In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema4.nc")

In [ ]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

In [4]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
import xarray as xr
import numpy as np
material_fractions = prep_data["material_fractions"]

# Create a new DataArray with the same shape and coordinates
maintenance_material = xr.DataArray(
    np.zeros_like(material_fractions),  # Change np.zeros_like to other values if needed
    coords=material_fractions.coords,   # Copying the same coordinates
    dims=material_fractions.dims,       # Copying the same dimension names
    name="maintenance_material"
)



In [12]:
material_fractions.to_netcdf("material_fractions.nc")

In [11]:
maintenance_material

<xarray.DataArray 'maintenance_material' (Cohort: 254, Type: 53, material: 14)> Size: 2MB
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
...
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]])
Coordinates:
  * Cohort    (Cohort) int64 2kB 1807 1808 1809 1810 ... 2057 2058 2059 2060
  * material  (material) <U9 504B 'Aluminium' 'Co' 'Cu' ... 'Steel' 'Ti' 'Wood'
  * Type      (Type) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'

In [7]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=False, 
    material=material)

In [ ]:
main_model_factory = ModelFactory(prep_data, complete_timeline).add(GenericStocks).add(GenericMaterials).finish()

In [ ]:
main_model_factory.simulate(simulation_timeline)

In [ ]:
main_model_normal.simulate(simulation_timeline)

In [ ]:
main_model_factory.stock_by_cohort.sum() - main_model_normal.stock_model.stock_by_cohort.sum()

<xarray.DataArray ()> Size: 8B
array(0.03125)

In [ ]:
assert (main_model_factory.stock_by_cohort == main_model_normal.stock_model.stock_by_cohort).all().values

AssertionError: 

In [ ]:
main_model_normal.material_model.stock_by_cohort_materials.sum()

<xarray.DataArray ()> Size: 8B
array(2.2558223e+15)

In [ ]:
# stocks_model.simulate(prism.Timeline(1970, 2060, 1))

In [ ]:
# main_model = ModelFactory(prep_data).add("stock").add("material").run()

In [ ]:
# main_model.outflow_by_cohort[2020].max()

In [ ]:
# model = simulate_stocks(prep_data)

In [ ]:
# export_summary_netcdf(model, "data_sums.nc")